The film industry tends to treat female-led stories as a commercial risk. That belief shapes budgets, and budgets shape what gets made. I wanted to see if the data backs it up.

The **Bechdel test** asks three questions of a film. Are there two named women? Do they talk to each other? Do they talk about something other than a man? It's a low bar, and passing says little about quality. But it's been applied the same way to thousands of films, so it's usable.

> **Research question.** Among commercially released US films from 1990 to 2015 with dialogue data, how does passing the Bechdel test relate to box office gross, at the same production budget and release year?

Every result here is an **association, not a cause**. Nothing below shows that changing a film's representation would change its revenue.

## The data

I built one table from three public sources:

| Source | What it has | How it was collected |
|---|---|---|
| **The Pudding** film dialogue | Word counts by character and gender, 2,000 films | Screenplays parsed by Hanah Anderson & Matt Daniels (2016). Gender coded from IMDb cast lists. |
| **bechdeltest.com** | Bechdel rating, 0–3 | Submitted and reviewed by volunteers. Accessed through the TidyTuesday 2021 mirror. |
| **The Numbers** | Production budget, domestic and worldwide gross | Industry reporting. Accessed through two static mirrors (TidyTuesday 2018, WWU DATA 311 2023). |

Two sources went offline during the project. bechdeltest.com shut down its API for good. The Numbers removed its public budget table in a site rebuild. I switched to mirrors for both and logged their limits in the project's provenance file.

Joining all three left **798 films**. Earlier notebooks handle the parsing, cleaning and tiered fuzzy title match. This page starts from the finished table.

In [1]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

INTERIM = Path.cwd().parent / "data" / "interim"

df = pd.read_csv(INTERIM / "analysis_table.csv")
spine = pd.read_csv(INTERIM / "spine.csv")         # after the Bechdel join
pud = pd.read_csv(INTERIM / "pudding_film.csv")    # before any join

# Variables reused throughout.
df["log_dom"] = np.log10(df["domestic_gross"])
df["log_ww"] = np.log10(df["worldwide_gross"])
df["log_budget"] = np.log10(df["production_budget"])
df["pass_"] = df["bechdel_pass"].astype(int)
df["pfd_10pp"] = df["pct_female_dialogue"] * 10   # 1 unit = 10 percentage points
df["year_c"] = df["year"] - 2000                  # centered for a readable intercept
df["result"] = np.where(df["bechdel_pass"], "Passes", "Fails")

PASS_FAIL = alt.Scale(domain=["Passes", "Fails"], range=["#3987e5", "#d95926"])

print(f"{len(df)} films x {df.shape[1]} columns, released {df['year'].min()}-{df['year'].max()}")
print(f"Bechdel pass rate: {df['bechdel_pass'].mean():.1%}")

798 films x 30 columns, released 1990-2015
Bechdel pass rate: 49.5%


### What the joins cost

Each source covers only some films, so every join drops films. I checked whether those drops were random. They weren't.

In [2]:
#| caption: Every stage of the merge kept the bigger films. Gross is the Pudding's inflation-adjusted domestic figure.
stages = {
    "1. Pudding films, 1990-2015": pud[pud["year"].between(1990, 2015)],
    "2. + Bechdel rating": spine,
    "3. + budget and gross": df,
}
funnel = pd.DataFrame({
    name: {
        "Films": len(d),
        "Mean gross ($M)": d["gross_infl_adj_musd"].mean(),
        "Median gross ($M)": d["gross_infl_adj_musd"].median(),
        "Female dialogue (%)": 100 * d["pct_female_dialogue"].mean(),
    }
    for name, d in stages.items()
}).T
funnel["Films"] = funnel["Films"].astype(int)
funnel.round(1)

,Films,Mean gross ($M),Median gross ($M),Female dialogue (%)
"1. Pudding films, 1990-2015",1587,92.9,50.0,29.0
2. + Bechdel rating,1189,105.9,62.0,30.0
3. + budget and gross,798,124.9,80.0,28.3


In [3]:
#| hide
#| caption: Mean inflation-adjusted domestic gross at each merge stage.
f = funnel.reset_index(names="stage")
f["label"] = f["stage"] + " · " + f["Films"].map("{:,} films".format)
base = alt.Chart(f).encode(
    y=alt.Y("stage:N", sort=None, axis=None),
    tooltip=[alt.Tooltip("stage:N", title="Stage"), alt.Tooltip("Films:Q", format=","),
             alt.Tooltip("Mean gross ($M):Q", format=".1f")],
)
bars = base.mark_bar(size=26).encode(
    x=alt.X("Mean gross ($M):Q", title="Mean gross, inflation-adjusted ($M)"))
values = bars.mark_text(align="left", dx=6, color="#e8e9ec").encode(
    text=alt.Text("Mean gross ($M):Q", format="$.0f"))
# Stage names sit above their bars, so long names never squeeze the plot.
names = base.mark_text(align="left", baseline="bottom", x=0, dy=-17,
                       font="JetBrains Mono, ui-monospace, monospace", fontSize=11).encode(
    text="label:N")
(bars + values + names).properties(title="Each join kept the larger films", height=210)

Mean inflation-adjusted gross rose from **\$93M to \$106M to \$125M** across the three stages. Half the films dropped out along the way. So this sample describes well-documented commercial releases, not films in general. I carry that limit through the rest of the analysis.

### Decision: put money on a log scale

The three money variables are heavily right-skewed. A few blockbusters sit far out in the tail. I applied a **base-10 log** to budget and both gross measures, for three reasons:

1. Box office is multiplicative. A film earns some *multiple* of its budget, so differences are proportional.
2. Without the log, a few blockbusters would dominate a least-squares fit.
3. On the log scale, a regression coefficient reads as a **percentage** difference. That's how I report the result.

I used `log10` instead of the natural log because one unit is one order of magnitude, which is easier to read. I didn't trim any films. The extreme values are real films, and they sit *on* the budget–gross trend, not off it.

In [4]:
money = ["production_budget", "domestic_gross", "worldwide_gross"]
pd.DataFrame({
    "Skew (raw $)": [df[c].skew() for c in money],
    "Skew (log10)": [np.log10(df[c]).skew() for c in money],
}, index=["Production budget", "Domestic gross", "Worldwide gross"]).round(2)

,Skew (raw $),Skew (log10)
Production budget,1.41,-0.77
Domestic gross,2.84,-1.07
Worldwide gross,3.66,-0.66


## The finding that wasn't there

I started with the obvious test: split films into pass and fail, then compare mean domestic gross.

- **H₀:** mean log₁₀ domestic gross is equal for films that pass and films that fail.
- **H₁:** the two means differ (two-tailed, α = 0.05).

**Why Welch's t-test?** It compares the means of one continuous variable across two independent groups. Welch's version doesn't assume equal variances. When the variances *are* equal, it loses almost no power. So it's the safer default.

**Do the assumptions hold? Partly.** Shapiro-Wilk rejects normality for both groups. Each has a tail of films that badly underperformed. But with about 400 films per group, the Central Limit Theorem makes the sampling distribution of the mean close to normal anyway. As a check, I also ran a **Mann-Whitney U test**, which doesn't assume normality.

In [5]:
grp_pass = df.loc[df["bechdel_pass"], "log_dom"]
grp_fail = df.loc[~df["bechdel_pass"], "log_dom"]

t_stat, p_t = stats.ttest_ind(grp_pass, grp_fail, equal_var=False)
U, p_u = stats.mannwhitneyu(grp_pass, grp_fail, alternative="two-sided")
diff_pct = 100 * (10 ** (grp_pass.mean() - grp_fail.mean()) - 1)

print(f"films:            {len(grp_pass)} pass, {len(grp_fail)} fail")
print(f"Welch's t-test:   t = {t_stat:.2f}, p = {p_t:.3f}")
print(f"Mann-Whitney U:   U = {U:,.0f}, p = {p_u:.3f}")
print(f"difference:       passing films earned {diff_pct:+.1f}% vs failing films")

films:            395 pass, 403 fail
Welch's t-test:   t = -0.35, p = 0.728
Mann-Whitney U:   U = 76,906, p = 0.409
difference:       passing films earned -3.1% vs failing films


In [6]:
#| hide
#| caption: The two distributions sit almost exactly on top of each other.
alt.Chart(df[["log_dom", "result"]]).transform_density(
    "log_dom", groupby=["result"], as_=["log_dom", "density"], extent=[4.5, 9.1],
).mark_area(opacity=0.45, line={"strokeWidth": 2}).encode(
    x=alt.X("log_dom:Q", title="Domestic gross (log10 dollars; 8 = $100M)"),
    y=alt.Y("density:Q", title="Density", stack=None),
    color=alt.Color("result:N", scale=PASS_FAIL, title=None),
    tooltip=[alt.Tooltip("result:N", title="Bechdel"),
             alt.Tooltip("log_dom:Q", title="log10 gross", format=".2f"),
             alt.Tooltip("density:Q", format=".3f")],
).properties(title="Domestic gross, passing vs failing films", height=260)

alt.Chart(...)

Nothing. Passing films earned 3.1% less (p = 0.728). Mann-Whitney agreed (p = 0.409). On its face, representation and revenue look unrelated.

Stopping here would have been a mistake.

## The finding that appeared

The correlations had already shown something the t-test couldn't. Budget correlates strongly with gross. That's expected: bigger films earn more. But budget also correlates *negatively* with Bechdel rating.

In [7]:
#| label: Correlation of budget with domestic gross (r)
r_budget_gross = df["production_budget"].corr(df["domestic_gross"])
round(r_budget_gross, 3)

np.float64(0.652)

In [8]:
median_budget = (df.groupby("bechdel_rating")["production_budget"]
                  .agg(films="size", median="median", mean="mean") / [1, 1e6, 1e6])
median_budget.columns = ["Films", "Median budget ($M)", "Mean budget ($M)"]
median_budget.index.name = "Bechdel rating"
median_budget.astype({"Films": int}).round(1)

,Films,Median budget ($M),Mean budget ($M)
Bechdel rating,,,
0,49,62.0,68.7
1,259,55.0,63.4
2,95,30.0,51.1
3,395,35.0,51.6


In [9]:
#| hide
#| caption: Box plots of production budget by Bechdel rating (3 = passes all three criteria). Hover for the quartiles.
alt.Chart(df[["bechdel_rating"]].assign(budget_m=df["production_budget"] / 1e6)).mark_boxplot(
    size=46, extent=1.5, median={"color": "#e8e9ec", "strokeWidth": 2},
    rule={"color": "#747a87"}, ticks=False,
    outliers={"size": 16, "opacity": 0.6, "color": "#86b6ef"},
).encode(
    x=alt.X("bechdel_rating:O", title="Bechdel rating (3 = passes)", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("budget_m:Q", title="Production budget ($M)"),
    color=alt.Color("bechdel_rating:O", legend=None,
                    scale=alt.Scale(range=["#1c5cab", "#2a78d6", "#5598e7", "#86b6ef"])),
).properties(title="Films with better representation are made more cheaply", height=320)

alt.Chart(...)

Films rated 0 have a median budget of **\$62M**. Films rated 3 have a median budget of **\$35M**.

That makes budget a **suppressor**. Passing films get less money, and less money means less revenue. That pull hides whatever else is going on. The raw comparison measured two opposing effects at once, and they roughly cancelled out.

So I ran the comparison again with budget and release year held fixed. Before fitting any model, I **wrote down the specification**. That way I couldn't pick whichever version gave the best answer.

> - **M1 (primary):** `log_dom ~ bechdel_pass + log_budget + year_c`
> - **M2 (secondary):** `log_dom ~ pct_female_dialogue + log_budget + year_c`
>
> **Estimator:** ordinary least squares with HC3 robust standard errors, since residual spread may not be constant across budgets. Release year absorbs price changes over the 26 years, so nominal dollars work as the outcome.
>
> **Four robustness checks, declared in advance:** worldwide gross, inflation-adjusted gross, rating as an ordinal factor, and genre controls. I fit no other models.

In [10]:
m1 = smf.ols("log_dom ~ pass_ + log_budget + year_c", data=df).fit(cov_type="HC3")
print(m1.summary().tables[1])
print(f"\nN = {int(m1.nobs)},  R2 = {m1.rsquared:.3f}")

b = m1.params["pass_"]
lo, hi = m1.conf_int().loc["pass_"]
pct = lambda v: 100 * (10 ** v - 1)    # log10 coefficient -> % difference in gross
print(f"\nBechdel pass: {pct(b):+.1f}% gross at fixed budget and year "
      f"(95% CI {pct(lo):+.1f}% to {pct(hi):+.1f}%, p = {m1.pvalues['pass_']:.3f})")

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.8572      0.306      6.078      0.000       1.258       2.456
pass_          0.0853      0.030      2.849      0.004       0.027       0.144
log_budget     0.7722      0.039     19.780      0.000       0.696       0.849
year_c        -0.0080      0.002     -3.552      0.000      -0.012      -0.004

N = 798,  R2 = 0.454

Bechdel pass: +21.7% gross at fixed budget and year (95% CI +6.3% to +39.3%, p = 0.004)


In [11]:
#| hide
#| label: Domestic gross, passing vs failing films, at the same budget and year
f'{pct(b):+.1f}%'

'+21.7%'

In [12]:
#| hide
#| caption: Point estimates with 95% confidence intervals. Holding budget and year fixed turns a null result into a clear positive one.
raw = smf.ols("log_dom ~ pass_", data=df).fit(cov_type="HC3")
compare = pd.DataFrame([
    {"model": "Raw comparison", "order": 1, "est": pct(raw.params["pass_"]),
     "lo": pct(raw.conf_int().loc["pass_", 0]), "hi": pct(raw.conf_int().loc["pass_", 1]),
     "p": raw.pvalues["pass_"]},
    {"model": "Budget and year fixed", "order": 0, "est": pct(b),
     "lo": pct(lo), "hi": pct(hi), "p": m1.pvalues["pass_"]},
])
base = alt.Chart(compare).encode(
    y=alt.Y("model:N", title=None, sort=alt.EncodingSortField("order")),
    color=alt.Color("model:N", legend=None,
                    scale=alt.Scale(domain=list(compare["model"]), range=["#747a87", "#3987e5"])),
    tooltip=[alt.Tooltip("model:N", title="Model"),
             alt.Tooltip("est:Q", title="Difference (%)", format="+.1f"),
             alt.Tooltip("lo:Q", title="95% CI low", format="+.1f"),
             alt.Tooltip("hi:Q", title="95% CI high", format="+.1f"),
             alt.Tooltip("p:Q", title="p-value", format=".3f")],
)
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[4, 4], color="#747a87").encode(x="x:Q")
ci = base.mark_rule(strokeWidth=3, strokeCap="round").encode(
    x=alt.X("lo:Q", title="Difference in gross, pass vs fail (%)"), x2="hi:Q")
dot = base.mark_point(size=180, filled=True, opacity=1)
txt = base.mark_text(dy=-16, fontWeight=600, fontSize=13).encode(
    x="est:Q", text=alt.Text("est:Q", format="+.1f"))
(zero + ci + dot.encode(x="est:Q") + txt).properties(
    title="The same comparison, before and after controlling for budget", height=170)

alt.LayerChart(...)

At the same budget and year, passing the Bechdel test is associated with **21.7% higher domestic gross** (95% CI +6.3% to +39.3%, p = 0.004).

Put plainly: **passing films earn more than failing films that cost the same.** They're just given less to work with.

### Does it hold up?

In [13]:
#| caption: The Bechdel-pass coefficient under each pre-declared check, as a % difference in gross.
models = {
    "M1: domestic (primary)": m1,
    "R1: worldwide": smf.ols("log_ww ~ pass_ + log_budget + year_c", data=df).fit(cov_type="HC3"),
}
r2df = df[df["gross_infl_adj_musd"] > 0].assign(log_adj=lambda d: np.log10(d["gross_infl_adj_musd"]))
models["R2: inflation-adjusted"] = smf.ols("log_adj ~ pass_ + log_budget + year_c",
                                                 data=r2df).fit(cov_type="HC3")
models["R4: + genre"] = smf.ols("log_dom ~ pass_ + log_budget + year_c + C(genre)",
                                         data=df.dropna(subset=["genre"])).fit(cov_type="HC3")

robust = pd.DataFrame([{
    "Model": name, "N": int(m.nobs),
    "Difference (%)": pct(m.params["pass_"]),
    "CI low (%)": pct(m.conf_int().loc["pass_", 0]),
    "CI high (%)": pct(m.conf_int().loc["pass_", 1]),
    "p": m.pvalues["pass_"],
} for name, m in models.items()]).set_index("Model")
robust.round({"Difference (%)": 1, "CI low (%)": 1, "CI high (%)": 1, "p": 3})

,N,Difference (%),CI low (%),CI high (%),p
Model,,,,,
M1: domestic (primary),798,21.7,6.3,39.3,0.004
R1: worldwide,798,20.9,5.5,38.5,0.006
R2: inflation-adjusted,787,18.5,4.2,34.9,0.010
R4: + genre,711,17.4,1.3,36.0,0.033


In [14]:
#| hide
#| caption: All four estimates sit above zero. The intervals are wide, so the direction is firmer than the size.
rb = robust.reset_index()
base = alt.Chart(rb).encode(
    y=alt.Y("Model:N", title=None, sort=None),
    tooltip=["Model", "N", alt.Tooltip("Difference (%):Q", format="+.1f"),
             alt.Tooltip("CI low (%):Q", format="+.1f"), alt.Tooltip("CI high (%):Q", format="+.1f"),
             alt.Tooltip("p:Q", format=".3f")],
)
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[4, 4], color="#747a87").encode(x="x:Q")
(zero
 + base.mark_rule(strokeWidth=3, strokeCap="round", color="#3987e5").encode(
     x=alt.X("CI low (%):Q", title="Difference in gross, pass vs fail (%)"), x2="CI high (%):Q")
 + base.mark_point(size=120, filled=True, opacity=1, color="#3987e5").encode(x="Difference (%):Q")
).properties(title="The association survives every pre-declared check", height=200)

alt.LayerChart(...)

It held across the pre-declared checks: worldwide gross (+20.9%), inflation-adjusted gross (+18.5%) and genre controls (+17.4%). Treating rating as an ordinal factor (R3) showed the association sits at the top ratings. It doesn't climb steadily from 0 to 3, which fits a threshold.

**The threshold matters, not the word count.** The secondary model replaces pass/fail with the *share* of dialogue spoken by women. That measure showed no association with revenue.

In [15]:
m2 = smf.ols("log_dom ~ pfd_10pp + log_budget + year_c", data=df).fit(cov_type="HC3")
b2 = m2.params["pfd_10pp"]
lo2, hi2 = m2.conf_int().loc["pfd_10pp"]
print(f"Per +10 percentage points of female dialogue: {pct(b2):+.1f}% gross "
      f"(95% CI {pct(lo2):+.1f}% to {pct(hi2):+.1f}%, p = {m2.pvalues['pfd_10pp']:.2f})")

Per +10 percentage points of female dialogue: +0.7% gross (95% CI -3.2% to +4.7%, p = 0.75)


A film can give women lots of dialogue without two of them ever talking to each other. The Bechdel *threshold* (two women talking about something other than a man) is what carries the association.

## Three more methods

Next I tried three other methods. Two pointed back to the same relationship. The third showed why the simple model was the right one.

### Can a film's packaging predict representation?

This flips the question. Can a film's **production characteristics** predict whether it passes? I used **logistic regression** because the target is binary and the coefficients are easy to interpret. The classes are almost evenly split, so accuracy is a fair metric.

**Predictors: genre, MPAA rating, log budget and release year.** I left out the dialogue measures on purpose. They measure how much women speak, so they'd predict well just by restating the outcome. Using only production characteristics keeps the question a real one. Categories are one-hot encoded and numbers standardized inside a pipeline. Scaling is fit on training folds only, so nothing leaks.

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

clf_df = df.dropna(subset=["genre", "mpaa_rating"])   # 87 films lack genre/MPAA
CAT, NUM = ["genre", "mpaa_rating"], ["log_budget", "year_c"]
clf = Pipeline([
    ("pre", ColumnTransformer([
        ("cat", OneHotEncoder(drop=["Drama", "R"], handle_unknown="ignore"), CAT),
        ("num", StandardScaler(), NUM),
    ])),
    ("logreg", LogisticRegression(max_iter=2000, random_state=42)),
])
cv = cross_val_score(clf, clf_df[CAT + NUM], clf_df["pass_"],
                     cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy")
baseline = max(clf_df["pass_"].mean(), 1 - clf_df["pass_"].mean())
print(f"{len(clf_df)} films; baseline (always guess the bigger class): {baseline:.1%}")
print(f"5-fold cross-validated accuracy: {cv.mean():.1%} +/- {cv.std():.1%}")

711 films; baseline (always guess the bigger class): 50.1%
5-fold cross-validated accuracy: 58.8% +/- 3.7%


In [17]:
#| hide
#| caption: Bechdel pass rate by genre. Hover for the number of films in each.
genre = (clf_df.groupby("genre")["bechdel_pass"].agg(rate="mean", films="size").reset_index())
bars = alt.Chart(genre).mark_bar(size=26).encode(
    x=alt.X("rate:Q", title="Share of films that pass", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 1])),
    y=alt.Y("genre:N", title=None, sort="-x"),
    tooltip=[alt.Tooltip("genre:N", title="Genre"), alt.Tooltip("rate:Q", title="Pass rate", format=".1%"),
             alt.Tooltip("films:Q", title="Films")],
)
half = alt.Chart(pd.DataFrame({"x": [0.5]})).mark_rule(strokeDash=[4, 4], color="#747a87").encode(x="x:Q")
lab = bars.mark_text(align="left", dx=6, color="#e8e9ec").encode(text=alt.Text("rate:Q", format=".0%"))
(bars + lab + half).properties(title="Horror passes at nearly double Action's rate", height=220)

alt.LayerChart(...)

The model reached **59% accuracy against a 50% baseline**. That near-miss is the result: **you can't tell whether a film represents women from its commercial profile.** Representation has to be measured, not inferred.

What little signal there is comes from category, not money. One genre stands out: **horror films pass at 74%, nearly double action's 38%.** Budget still has a negative coefficient after accounting for genre and rating. It's the same relationship as before, showing up in a different model.

### Do films cluster by representation on their own?

I ran **k-means** on four variables: log budget, log gross, female dialogue share and female character share. I standardized them first. k-means works on distance, and the variables use different units, so without scaling the widest one would decide everything. I **left Bechdel rating out** so I could check the clusters against it afterward. Including it would make that check circular.

**Choosing k.** Silhouette score peaks at k = 2 (0.355), with k = 3 close behind (0.324). I chose **k = 3** for interpretability. Two clusters force budget and representation onto one axis. Three keeps them apart.

In [18]:
from sklearn.cluster import KMeans

FEATURES = ["log_budget", "log_dom", "pct_female_dialogue", "pct_female_characters"]
X_clust = StandardScaler().fit_transform(df[FEATURES])
df["cluster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_clust).labels_

names = {0: "Small budget, male-dominated", 1: "Big budget, male-dominated",
         2: "Mid budget, female-forward"}
df["cluster_name"] = df["cluster"].map(names)

summary = df.groupby("cluster_name").agg(
    Films=("cluster", "size"),
    **{"Budget ($M)": ("production_budget", lambda s: s.mean() / 1e6),
       "Female dialogue %": ("pct_female_dialogue", lambda s: 100 * s.mean()),
       "Pass rate %": ("bechdel_pass", lambda s: 100 * s.mean())},
).sort_values("Pass rate %")
summary.index.name = "Cluster"
summary.round(1)

,Films,Budget ($M),Female dialogue %,Pass rate %
Cluster,,,,
"Big budget, male-dominated",424,83.8,18.2,36.1
"Small budget, male-dominated",192,15.7,23.6,43.8
"Mid budget, female-forward",182,35.7,56.6,86.8


In [19]:
#| hide
#| caption: Each dot is a film, coloured by the cluster k-means assigned without seeing Bechdel ratings. Hover for the title.
films = df[["title", "year", "pct_female_dialogue", "result", "cluster_name"]]
alt.Chart(films.assign(budget_m=df["production_budget"] / 1e6)).mark_circle(size=40, opacity=0.75).encode(
    x=alt.X("budget_m:Q", title="Production budget ($M, log scale)", scale=alt.Scale(type="log", domain=[0.4, 320])),
    y=alt.Y("pct_female_dialogue:Q", title="Share of dialogue spoken by women", axis=alt.Axis(format="%")),
    color=alt.Color("cluster_name:N", title=None, scale=alt.Scale(
        domain=[names[0], names[1], names[2]], range=["#3987e5", "#d95926", "#199e70"])),
    tooltip=[alt.Tooltip("title:N", title="Film"), alt.Tooltip("year:Q", title="Year"),
             alt.Tooltip("budget_m:Q", title="Budget ($M)", format=".1f"),
             alt.Tooltip("pct_female_dialogue:Q", title="Female dialogue", format=".0%"),
             alt.Tooltip("result:N", title="Bechdel"), alt.Tooltip("cluster_name:N", title="Cluster")],
).properties(title="Three clusters, found without the Bechdel ratings", height=360)

alt.Chart(...)

Checked against Bechdel ratings afterward, the clusters' pass rates were **36%, 44% and 87%**. Two separately produced measures of representation agree: the dialogue data and the volunteer ratings.

The female-forward cluster is mid-budget films where women speak 57% of the dialogue. Money separates the other two clusters almost entirely, and both give women under a quarter of the lines. **Small, cheap films aren't automatically female-forward.** That surprised me most.

### Does a more flexible model do better?

**Random forest** was the one method that added nothing. I re-ran the gross-prediction question with it. I used the same predictors and the same five cross-validation folds as ordinary least squares, so only the algorithm changed.

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

ML = ["log_budget", "year_c", "pct_female_dialogue", "pct_female_characters", "pass_"]
folds = KFold(n_splits=5, shuffle=True, random_state=42)    # identical folds for both
ols = Pipeline([("scaler", StandardScaler()), ("ols", LinearRegression())])
forest = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)

scores = {name: cross_val_score(model, df[ML], df["log_dom"], cv=folds, scoring="r2")
          for name, model in [("OLS", ols), ("Random forest", forest)]}
pd.DataFrame({"Mean R²": {k: v.mean() for k, v in scores.items()},
              "Std R²": {k: v.std() for k, v in scores.items()}}).round(4)

,Mean R²,Std R²
OLS,0.4406,0.1020
Random forest,0.3655,0.1083


It scored worse than ordinary least squares (R² 0.366 vs 0.441). After the log transform, the relationship is close to linear, so the extra flexibility bought nothing. Here the linear model isn't a simplification. It's the right specification.

One more contrast. In the forest, the Bechdel variable has almost no *feature importance*, yet the regression found it significant. Both are right because they answer different questions. Importance asks how much a variable helps predict an unknown film's gross. The answer is barely, since budget explains most of what's predictable. The regression asks how much passing films differ *from films with the same budget*. The answer is meaningfully. Prediction and explanation are different goals.

## Recommendations

**For studios and financiers.** Passing films get smaller budgets, and this data doesn't support that gap. At the same budget, passing films earned about a fifth more. That doesn't prove a bigger budget would pay off for any one film. But if the smaller budgets rest on a belief that these films earn less, this sample doesn't back it up.

**Don't infer representation from commercial signals.** Genre, rating, budget and year predict Bechdel results barely better than chance. The money variables are the weakest of the four. Any process that infers representation from how a film is packaged is guessing. Measure it instead.

**Look at interaction, not word count.** Female dialogue *share* showed no link to revenue (+0.7% per 10 points, p = 0.75). The Bechdel *threshold* did. Two women talking to each other is what tracks with revenue, not the total amount women speak.

**For analysts.** Check for suppression before you report a null. My unadjusted t-test pointed to the opposite of what the data shows.

## What this does not show

This is association, not causation. Nothing here shows that changing a script would change revenue. Passing and failing films likely differ in ways budget and year don't capture, like casting, marketing spend or director experience.

- **The sample skews toward larger commercial releases** because of the three-way join. That narrows the range of budgets and grosses. Range restriction usually weakens an estimate rather than inflating it, but I haven't tested that here.
- **Ratings come from volunteers**, not trained coders. The 2021 mirror doesn't flag disputed entries.
- **Gender coding is binary.** It erases non-binary identities and adds error for characters with ambiguous names.
- **The Bechdel test is coarse.** A film can pass on one throwaway exchange. A great film about one woman has to fail.
- **The confidence interval is wide** (+6.3% to +39.3%). The direction is firmer than the size.

The claim I'll defend is narrow: **in this sample, the commercial penalty for representation shows up in what films are given, not in what they earn.**

## Sources and full notebooks

This page is a short version of the project. The full pipeline (ingesting the sources, the tiered fuzzy title match, every assumption check and diagnostic) is in the [project's notebooks on GitHub](https://github.com/mckeeander/mckeeander.github.io/tree/main/projects/bechdel).

- Anderson, H., & Daniels, M. (2016). *Film dialogue from 2,000 screenplays, broken down by gender and age.* [The Pudding](https://pudding.cool/2017/03/film-dialogue/).
- Bechdel Test Movie List (2021). Bechdel test data, via the TidyTuesday 2021-03-09 mirror. [bechdeltest.com](https://bechdeltest.com/), CC BY-NC 3.0.
- Nash Information Services (2018, 2023). Movie budgets and financial performance records, via the TidyTuesday 2018-10-23 snapshot and the WWU DATA 311 course mirror. [The Numbers](https://www.the-numbers.com/).

*DSC 530 Data Exploration and Analysis, Summer 2026.*